# vLLM Chat + CrewAI (Modal Notebook)

**Flow:** Create a vLLM endpoint (see **Create an endpoint** below), then run Step 1 to connect. Steps 2–3 use the OpenAI client for chat; Step 4 runs a small CrewAI agent with the same endpoint.

---
## Create an endpoint (do this first)

**Fast path (avoids long "creating objects" in the notebook):** Use **Option A**. Run `modal deploy vllm_modal_serve.py` once in a terminal (first time ~10–15 min; then cached). Add the printed URL to Secrets as **OPENAI_BASE_URL**. The notebook then just connects—no heavy image build in the notebook.

This notebook needs a vLLM endpoint. **Choose one:**

### Option A — Deploy a separate vLLM server (recommended)

1. In a terminal, from the `class3_runs` folder run:
   ```bash
   modal deploy vllm_modal_serve.py
   ```
2. Copy the **URL** printed at the end (e.g. `https://you--vllm-chat-demo-serve.modal.run`).
3. In this Modal notebook: **Settings** (gear) → **Secrets** → **Add secret** → create a secret with:
   - **OPENAI_BASE_URL** = that URL + `/v1`  
   - Example: `https://you--vllm-chat-demo-serve.modal.run/v1`
4. Re-run **Step 1** below. The notebook will use this endpoint; no need for the vllm-notebook image.

### Option B — Run vLLM inside this notebook

1. In a terminal run:
   ```bash
   modal deploy vllm_notebook_image.py
   ```
2. In this Modal notebook: **Settings** → **Image** → search **vllm-notebook** → select it. Use a **GPU** (e.g. A10G).
3. Re-run **Step 1** below. It will start the vLLM server in a subprocess in this kernel.

### Where to add the endpoint URL

**Option 1 — Modal Secrets (recommended):**  
In the notebook toolbar: **Settings** (gear icon) → **Secrets** → **Add secret** → add `OPENAI_BASE_URL` = your URL **with `/v1`** at the end (e.g. `https://you--vllm-chat-demo-serve.modal.run/v1`). Then run Step 1.

**Option 2 — Set in code below:**  
Run the cell below and paste your endpoint URL (the one printed when you ran `modal deploy vllm_modal_serve.py`). Add `/v1` at the end. Then run Step 1.

In [ ]:
# Paste your endpoint URL here (from "modal deploy vllm_modal_serve.py" output). Must end with /v1.
# If you added OPENAI_BASE_URL in Settings → Secrets, leave the string below empty.
import os
ENDPOINT_URL = "https://goabiaryan--vllm-chat-demo-serve.modal.run/v1"  # from modal deploy vllm_modal_serve.py
if ENDPOINT_URL:
    url = ENDPOINT_URL.strip()
    if not url.endswith("/v1"):
        url = url.rstrip("/") + "/v1"
    os.environ["OPENAI_BASE_URL"] = url
    print("Endpoint set to:", url, "— Now run Step 1 below.")
else:
    print("Using OPENAI_BASE_URL from Secrets (or run Step 1 to connect).")

---
## Step 1 — Connect to vLLM

Run the cell below after you have an endpoint (see **Create an endpoint** above). It uses the deployed **vllm-chat-demo** app if found, else **OPENAI_BASE_URL**, else starts a vLLM server in this notebook (only if the image has vLLM, e.g. vllm-notebook).

In [ ]:
# Step 1 — Connect to vLLM (one cell, no vLLM import)
%pip install openai modal -q
import os
import sys
import time
import subprocess
from openai import OpenAI

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
VLLM_PORT = 8000
VLLM_BASE_URL = None

# 1) Deployed Modal app
try:
    import modal
    serve = modal.Function.from_name("vllm-chat-demo", "serve")
    VLLM_BASE_URL = serve.get_web_url() + "/v1"
    print("Using deployed vllm-chat-demo app.", flush=True)
except Exception:
    pass

# 2) OPENAI_BASE_URL env
if not VLLM_BASE_URL:
    VLLM_BASE_URL = os.environ.get("OPENAI_BASE_URL", "").strip() or None
    if VLLM_BASE_URL:
        print("Using OPENAI_BASE_URL.", flush=True)

# 3) Start vLLM server in background (subprocess) — only if this session has a GPU
if not VLLM_BASE_URL:
    has_gpu = False
    try:
        import torch
        has_gpu = torch.cuda.is_available()
    except Exception:
        pass
    if not has_gpu:
        print("No GPU in this notebook. To get an endpoint:", flush=True)
        print("  1. Run: modal deploy vllm_modal_serve.py  (from class3_runs)", flush=True)
        print("  2. In Modal Notebook → Settings → Secrets, add OPENAI_BASE_URL = <url from deploy> + /v1", flush=True)
        print("  3. Re-run this cell.", flush=True)
    else:
        VLLM_BASE_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"
        print("Starting vLLM server (subprocess)...", flush=True)
        proc = subprocess.Popen(
            [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
             "--host", "127.0.0.1", "--port", str(VLLM_PORT), "--dtype", "bfloat16", "--max-model-len", "8192"],
            env=os.environ, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        for _ in range(300):
            try:
                c = OpenAI(base_url=VLLM_BASE_URL, api_key="x", max_retries=0, timeout=2)
                c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=2)
                print("vLLM server ready.", flush=True)
                break
            except Exception:
                if proc.poll() is not None:
                    VLLM_BASE_URL = None
                    print("vLLM server exited (e.g. not installed). On Modal A10: use Image → vllm-notebook (run 'modal deploy vllm_notebook_image.py' first).", flush=True)
                    break
                time.sleep(2)
        else:
            VLLM_BASE_URL = None
            if has_gpu:
                print("Server did not become ready in time. If on Modal, ensure Image is vllm-notebook.", flush=True)

client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed") if VLLM_BASE_URL else None
print("Model:", MODEL, "| Endpoint:", VLLM_BASE_URL or "(not set)")

In [ ]:
# Client is ready from Step 1 above. Run the cells below for chat and CrewAI.
if not client:
    print("No endpoint yet. Either: (1) Use a GPU notebook and re-run Step 1, or (2) Run 'modal deploy vllm_modal_serve.py', set OPENAI_BASE_URL in Modal Notebook secrets to the deploy URL + /v1, then re-run Step 1.")

---
## Step 2 — Single-turn chat

One user message, one completion. Same pattern as the OpenAI Chat Completions API.

In [ ]:
# Single-turn chat (OpenAI client → vLLM endpoint)
if not client:
    print("No endpoint. Run Step 1 first. On Modal: use Image → vllm-notebook (run 'modal deploy vllm_notebook_image.py') or set OPENAI_BASE_URL to a deployed vllm-chat-demo URL.")
else:
    messages = [{"role": "user", "content": "In one short sentence, what is vLLM?"}]
    r = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=128, temperature=0.3)
    reply = r.choices[0].message.content
    print("User:", messages[0]["content"])
    print("Assistant:", reply)

---
## Step 3 — Multi-turn chat

Build a conversation by appending each user message and the assistant's reply to `messages`. The model sees the full history and can refer back to earlier turns.

In [ ]:
# Multi-turn chat (client → vLLM endpoint)
messages = []

def chat_turn(user_text: str) -> str:
    messages.append({"role": "user", "content": user_text})
    r = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=128, temperature=0.3)
    reply = r.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

# Turn 1
r1 = chat_turn("My name is Alex. Remember it.")
print("Turn 1 — User: My name is Alex. Remember it.")
print("Turn 1 — Assistant:", r1)
print()

# Turn 2: model should use the remembered name
r2 = chat_turn("What is my name?")
print("Turn 2 — User: What is my name?")
print("Turn 2 — Assistant:", r2)

---
## Step 4 — Small agent with CrewAI

CrewAI lets you define **agents** (role, goal, backstory) and **tasks**. We wire a single agent to use vLLM by passing an `LLM` with `base_url` pointing at the vLLM server. Run the cell below (it installs CrewAI if needed).

**CrewAI needs an OpenAI-compatible endpoint.** Run the cell below to use a deployed Modal app, `OPENAI_BASE_URL`, or to start a vLLM server in this kernel. Then run the CrewAI cells.

In [ ]:
%pip install crewai -q
from crewai import Agent, Task, Crew, LLM

if not VLLM_BASE_URL:
    print("No endpoint. Run Step 1 first. On Modal: use Image → vllm-notebook or set OPENAI_BASE_URL.")
else:
    # LLM backed by vLLM (OpenAI-compatible)
    llm = LLM(
        model=MODEL,
        base_url=VLLM_BASE_URL.rstrip("/"),  # e.g. http://localhost:8000/v1
        api_key=os.environ.get("OPENAI_API_KEY", "not-needed"),
        temperature=0.3,
    )

    # One agent: summarizer
    summarizer = Agent(
        role="Summarizer",
        goal="Give brief, clear summaries.",
        backstory="You are a concise assistant who summarizes text in 1–2 sentences.",
        llm=llm,
        verbose=True,
    )

    # Task: summarize a short paragraph
    task = Task(
        description="Summarize the following in one or two sentences: "
        "vLLM is an open-source inference engine for large language models. "
        "It supports high throughput and low latency and is compatible with the OpenAI API.",
        expected_output="A 1–2 sentence summary.",
        agent=summarizer,
    )

    crew = Crew(agents=[summarizer], tasks=[task])
    print("Running CrewAI agent (vLLM backend)...", flush=True)
    result = crew.kickoff()
    print("\nResult:", result)

In [ ]:
# Get or start server for CrewAI (and tuning). Run this before the CrewAI cells.
import sys
import time
import subprocess

if VLLM_BASE_URL is None:
    %pip install openai modal -q
    import modal
    from openai import OpenAI
    VLLM_PORT = 8000
    try:
        serve = modal.Function.from_name("vllm-chat-demo", "serve")
        VLLM_BASE_URL = serve.get_web_url() + "/v1"
        print("Using deployed vllm-chat-demo app.", flush=True)
    except Exception:
        pass
    if VLLM_BASE_URL is None:
        VLLM_BASE_URL = os.environ.get("OPENAI_BASE_URL", "").strip() or None
        if VLLM_BASE_URL:
            print("Using OPENAI_BASE_URL.", flush=True)
    if VLLM_BASE_URL is None:
        VLLM_BASE_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"
        print("Starting vLLM server in this kernel (may take a few min)...", flush=True)
        proc = subprocess.Popen(
            [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
             "--host", "127.0.0.1", "--port", str(VLLM_PORT), "--dtype", "bfloat16", "--max-model-len", "8192"],
            env=os.environ,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        for _ in range(300):
            try:
                c = OpenAI(base_url=VLLM_BASE_URL, api_key="x", max_retries=0, timeout=2)
                c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=2)
                print("vLLM server ready.", flush=True)
                break
            except Exception:
                if proc.poll() is not None:
                    print("vLLM server exited.", flush=True)
                    VLLM_BASE_URL = None
                    break
                time.sleep(2)
        else:
            print("Server did not become ready in time.", flush=True)
            VLLM_BASE_URL = None
    client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed") if VLLM_BASE_URL else None
else:
    from openai import OpenAI
    client = OpenAI(base_url=VLLM_BASE_URL, api_key="not-needed") if VLLM_BASE_URL else None

print("Endpoint:", VLLM_BASE_URL or "(not set)")

---
## Optional: Two agents (researcher + writer)

A minimal **crew** with two agents: one researches a topic, the other writes a short report. Both use vLLM.

In [ ]:
researcher = Agent(
    role="Researcher",
    goal="List 3 key points about a given topic.",
    backstory="You quickly identify the most important facts.",
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Writer",
    goal="Turn bullet points into a short paragraph.",
    backstory="You write clear, brief paragraphs.",
    llm=llm,
    verbose=True,
)

topic = "benefits of open-source LLM inference"

task_research = Task(
    description=f"List 3 key points about: {topic}",
    expected_output="3 bullet points.",
    agent=researcher,
)

task_write = Task(
    description="Turn the previous agent's bullet points into one short paragraph.",
    expected_output="One short paragraph.",
    agent=writer,
    context=[task_research],  # writer sees researcher's output
)

crew2 = Crew(agents=[researcher, writer], tasks=[task_research, task_write])
print(f"Crew: research + write on '{topic}'", flush=True)
result2 = crew2.kickoff()
print("\nFinal result:", result2)

---
## vLLM tuning: 6 knobs

Run the cells below to see default vs recommended settings (printed from code) and to compare baseline vs tuned benchmark results.

In [ ]:
# 6 knobs: default (before tuning) vs recommended
VLLM_KNOBS = [
    ("max_num_batched_tokens", "2048", "8192-32768 (throughput) or 4096 (latency)", "Biggest throughput lever; default is ITL-optimised."),
    ("gpu_memory_utilization", "0.90", "0.95", "Leaves 10% VRAM idle."),
    ("max_num_seqs", "256 (V0) / 1024 (V1)", "512-2048", "Hard caps concurrency; low = silent queueing."),
    ("enable_prefix_caching", "OFF", "ON", "Free win for shared system prompts / RAG."),
    ("enable_chunked_prefill", "OFF (V0) / ON (V1)", "ON", "Verify on V1."),
    ("cpu_cores", "often underprovisioned", "2 + num_GPUs", "Starving CPU → GPU idle."),
]
print("vLLM settings BEFORE tuning (defaults):", flush=True)
for knob, default, rec, why in VLLM_KNOBS:
    print(f"  {knob}: default={default}  -> recommended={rec}  ({why})", flush=True)
print(flush=True)

In [ ]:
# Presets: throughput-heavy vs latency-sensitive (used in vllm_modal_serve.py)
VLLM_PRESETS = {
    "default": {"description": "Out-of-the-box", "flags": []},
    "throughput_tuned": {
        "description": "Throughput-heavy prod",
        "flags": ["--max-num-batched-tokens", "16384", "--gpu-memory-utilization", "0.95", "--enable-prefix-caching", "--enable-chunked-prefill"],
    },
    "latency_tuned": {
        "description": "Latency-sensitive prod",
        "flags": ["--max-num-batched-tokens", "4096", "--max-num-seqs", "512", "--enable-chunked-prefill"],
    },
}

print("Recommended presets (from code):", flush=True)
print("  Throughput-heavy: max-num-batched-tokens 16384, gpu-memory-utilization 0.95, enable-prefix-caching, enable-chunked-prefill", flush=True)
print("  Latency-sensitive: max-num-batched-tokens 4096, max-num-seqs 512, enable-chunked-prefill", flush=True)
print(flush=True)

def vllm_serve_command(model: str, preset: str, port: int = 8000) -> str:
    base = f"python -m vllm.entrypoints.openai.api_server --model {model} --host 0.0.0.0 --port {port}"
    fl = VLLM_PRESETS.get(preset, {}).get("flags", [])
    return base + (" " + " ".join(fl) if fl else "")

print("Full serve commands:", flush=True)
for name, data in VLLM_PRESETS.items():
    print(f"  {name}: {data['description']}", flush=True)
    print(f"    {vllm_serve_command(MODEL, name)}", flush=True)
    print(flush=True)

In [ ]:
import time

def run_simple_benchmark(client, model, n_runs=5, max_tokens=64, prompt="List the numbers 1 to 5, one per line."):
    messages = [{"role": "user", "content": prompt}]
    total_times, ttft_times, token_counts = [], [], []
    for _ in range(n_runs):
        start = time.perf_counter()
        ttft_s = None
        try:
            stream = client.chat.completions.create(model=model, messages=messages, max_tokens=max_tokens, temperature=0.0, stream=True)
            first, n = True, 0
            for chunk in stream:
                if first and chunk.choices and chunk.choices[0].delta.content:
                    ttft_s = time.perf_counter() - start
                    first = False
                if chunk.choices and chunk.choices[0].delta.content:
                    n += 1
            total_times.append(time.perf_counter() - start)
            token_counts.append(n)
            if ttft_s is not None:
                ttft_times.append(ttft_s)
        except Exception as e:
            print(f"Request failed: {e}", flush=True)
    if not total_times:
        return {"error": "No successful runs"}
    n_tok = sum(token_counts) / len(token_counts)
    t_mean = sum(total_times) / len(total_times)
    out = {"tokens_per_sec_mean": round(n_tok / t_mean, 2), "total_time_s_mean": round(t_mean, 3), "completion_tokens_mean": round(n_tok, 1), "n_runs": len(total_times)}
    if ttft_times:
        out["ttft_s_mean"] = round(sum(ttft_times) / len(ttft_times), 3)
    return out

print("Running benchmark (current server config)...", flush=True)
results_baseline = run_simple_benchmark(client, MODEL, n_runs=5, max_tokens=64)
if "error" in results_baseline:
    print(results_baseline["error"], flush=True)
else:
    print("Baseline (current server):", results_baseline, flush=True)

---

Next: run the cell below after redeploying with tuned flags (see print output).

In [ ]:
print("To get tuned results:", flush=True)
print("  1. In vllm_modal_serve.py set THROUGHPUT_TUNED = True", flush=True)
print("  2. Run: modal deploy vllm_modal_serve.py", flush=True)
print("  3. Run the cell below to benchmark and compare.", flush=True)
print(flush=True)

In [ ]:
print("Running benchmark (tuned server config)...", flush=True)
results_tuned = run_simple_benchmark(client, MODEL, n_runs=5, max_tokens=64)
if "error" in results_tuned:
    print(results_tuned["error"], flush=True)
else:
    print("Tuned:", results_tuned, flush=True)

if "results_baseline" in dir() and "error" not in results_baseline and "error" not in results_tuned:
    b = results_baseline.get("tokens_per_sec_mean", 0) or 1e-6
    t = results_tuned.get("tokens_per_sec_mean", 0)
    print(f"\nThroughput: baseline {b:.1f} tok/s -> tuned {t:.1f} tok/s ({(t/b - 1) * 100:+.0f}%)", flush=True)
    print(f"Latency:    baseline {results_baseline.get('total_time_s_mean', 0):.2f}s -> tuned {results_tuned.get('total_time_s_mean', 0):.2f}s", flush=True)